# RAG Pipeline - Exp4

* About
  * Code is the same as Exp3, but changed pipeline settings a bit:
    * LLM changed from `llama-3.1-8b-instant` to `openai/gpt-oss-120b`
    * `alpha` for retriver increased from 0.6 to 0.9 (increased semantic indexing weight)
* Learnings & Observations 🍀
  * Human labeled referenced documents referenced answers can have problems or go out-of-date soon, therefore
    * auto eval better not to assume every record has ground truth
    * in the eval metrics better to include the score than considers AI's output is better than the reference to the query

In [1]:
%load_ext autoreload
%autoreload 2

import os
import yaml
os.environ.pop("OPENAI_API_KEY", None)

from datasets import load_dataset
from llama_index.core import Document
import pandas as pd
import json

from utils import *

import warnings
warnings.filterwarnings('ignore')

resource module not available on Windows


### Load Data

* This applies to any dataset that has:
  * `contexts`: ground truth referenced documents, list format
  * `ground_truths`, list format
  * `question`

In [2]:
fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval")['baseline']

output_dir = "fiqa_raw_text"
os.makedirs(output_dir, exist_ok=True)

rag_lst = []
documents = []  # to store documents for llamindex
for idx, record in enumerate(fiqa_eval):
    context = ''.join(record['contexts'])
    gt = ''.join(record['ground_truths'])
    if 'answer' in record.keys():
        ai0_answer = record['answer'].strip()
    else:
        ai0_answer = None

    with open(os.path.join(output_dir, f"{idx}.txt"), "w", encoding="utf-8") as f:
        f.write(context)
    rag_lst.append({
        'question': record['question'],
        'context': context,
        'context_ct': len(record['contexts']),
        'ground_truth': gt,
        'ai0_answer': ai0_answer
    })
    doc = Document(
        text=context,
        metadata={
            "doc_name": idx
        }
    )
    documents.append(doc)

rag_df = pd.DataFrame(rag_lst)
print(rag_df.shape, max(rag_df['context_ct']), min(rag_df['context_ct']))
print(len(documents))
rag_df.head()

(30, 5) 1 1
30


,question,context,context_ct,ground_truth,ai0_answer
0,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,1,Have the check reissued to the proper payee.Ju...,The best way to deposit a cheque issued to an ...
1,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,1,Sure you can. You can fill in whatever you wa...,"Yes, you can send a money order from USPS as a..."
2,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,1,You're confusing a lot of things here. Company...,"Yes, it is possible to have one EIN doing busi..."
3,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,1,"""I'm afraid the great myth of limited liabilit...",Applying for and receiving business credit can...
4,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,1,You should probably consult an attorney. Howev...,If your employer has closed and you need to tr...


### RAG Pipeline

In [3]:
with open("all_rag_pipeline_config.yaml", "r", encoding="utf-8") as f:
     all_config= yaml.safe_load(f)

In [4]:
for i in range(len(all_config['embedding_models'])):
    embed_model_str = all_config['embedding_models'][i]
    indexing_storage_dir = all_config['indexing_storage_dirs'][i]
    print(f"Embedding model: {embed_model_str}")
    create_vector_index(documents, embed_model_str, indexing_storage_dir)

Embedding model: BAAI/bge-small-en-v1.5


Parsing nodes:   0%|          | 0/30 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/35 [00:00<?, ?it/s]

Index saved to ./storage_m2


In [5]:
retriever_params = all_config['retriever_params']
cfgs = []
for i in range(len(all_config['yaml_config_files'])):
    yaml_config_file = 'output/saved_configs/' + all_config['yaml_config_files'][i]
    cfgs.append(yaml_config_file)
    config = {
        'llm_model': all_config['llm_models'][i],
        'embedding_model': all_config['embedding_models'][i],
        'indexing_storage_dir': all_config['indexing_storage_dirs'][i],
        'output_file': all_config['output_files'][i],
        'retriever_params': retriever_params
    }

    with open(yaml_config_file, "w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False)

In [6]:
max_worker = os.cpu_count() - 1

await run_all_in_processes(cfgs, rag_lst, documents, max_workers=max_worker)

### Evaluation

In [7]:
from langchain_groq import ChatGroq
import nest_asyncio
nest_asyncio.apply()

with open('eval_prompts.yaml', 'r') as file:
    prompt_versions = yaml.safe_load(file)

eval_llm = ChatGroq(
    groq_api_key=os.environ["GROQ_TOKEN"],
    model_name="openai/gpt-oss-20b", 
    temperature=0.7
)

eval_input_lst = []
for config in all_config['yaml_config_files']:
    yaml_config_file = 'output/saved_configs/' + config
    with open(yaml_config_file, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    output_file = config['output_file']
    response_lst = json.load(open(output_file, "r", encoding="utf-8"))
    eval_input = get_eval_input(response_lst)
    eval_input = pd.merge(eval_input, rag_df[['question', 'context']], left_on='query', right_on='question')
    eval_input.drop(columns=['question'], inplace=True)
    eval_input_lst.append(eval_input)
    print(eval_input.shape)
    display(eval_input.head())

(30, 5)


,query,ai_answer,referenced_answer,retrieved_content,context
0,How to deposit a cheque issued to an associate...,To deposit a check that’s made out to an assoc...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,Just have the associate sign the back and then...
1,Can I send a money order from USPS as a business?,Yes. The USPS money order lets you put any nam...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...
2,1 EIN doing business under multiple business n...,Yes. A single employer‑identification number i...,You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...
3,Applying for and receiving business credit,"To obtain business credit, follow these steps:...","""I'm afraid the great myth of limited liabilit...",Set up a meeting with the bank that handles yo...,Set up a meeting with the bank that handles yo...
4,401k Transfer After Business Closure,"When your employer shuts down, the 401(k) plan...",You should probably consult an attorney. Howev...,The time horizon for your 401K/IRA is essentia...,The time horizon for your 401K/IRA is essentia...


In [ ]:
for idx, eval_input in enumerate(eval_input_lst):
    retrieval_quality = asyncio.run(get_retrieval_quality_output_async(eval_input, eval_llm,
                                                            prompt_versions['rq_prompt_template']))
    retrieval_quality['same_context'] = retrieval_quality['retrieved_content'] == retrieval_quality['context']
    retrieval_quality.to_excel(all_config['eval_rq_output'][idx], index=False)
    print(retrieval_quality['retrieval_quality_score'].value_counts())
    print(retrieval_quality['same_context'].value_counts())
    display(retrieval_quality.head())

retrieval_quality_score
3    27
0     2
2     1
Name: count, dtype: int64
same_context
True     23
False     7
Name: count, dtype: int64


,query,ai_answer,referenced_answer,retrieved_content,context,retrieval_quality_score,rq_reasoning,same_context
0,How to deposit a cheque issued to an associate...,To deposit a check that’s made out to an assoc...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,3,The retrieved content is identical to the cont...,True
1,Can I send a money order from USPS as a business?,Yes. The USPS money order lets you put any nam...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,3,The retrieved content directly answers the use...,True
2,1 EIN doing business under multiple business n...,Yes. A single employer‑identification number i...,You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...,3,The retrieved content is identical to the prov...,True
3,Applying for and receiving business credit,"To obtain business credit, follow these steps:...","""I'm afraid the great myth of limited liabilit...",Set up a meeting with the bank that handles yo...,Set up a meeting with the bank that handles yo...,3,The retrieved content is identical to the cont...,True
4,401k Transfer After Business Closure,"When your employer shuts down, the 401(k) plan...",You should probably consult an attorney. Howev...,The time horizon for your 401K/IRA is essentia...,The time horizon for your 401K/IRA is essentia...,3,The retrieved content is a near‑duplicate of t...,False


In [ ]:
for idx, eval_input in enumerate(eval_input_lst):
    answer_quality = asyncio.run(get_answer_quality_output_async(eval_input, eval_llm,
                                                            prompt_versions['aq_prompt_template']))
    answer_quality.to_excel(all_config['eval_aq_output'][idx], index=False)
    print(answer_quality.shape)
    print(answer_quality['answer_quality_score'].value_counts(normalize=True))
    display(answer_quality.head())

(30, 7)
answer_quality_score
 4    0.400000
 2    0.233333
 3    0.166667
-1    0.100000
 0    0.066667
 1    0.033333
Name: proportion, dtype: float64


,query,ai_answer,referenced_answer,retrieved_content,context,answer_quality_score,aq_reasoning
0,How to deposit a cheque issued to an associate...,To deposit a check that’s made out to an assoc...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,4,The AI’s answer is fully relevant to the user’...
1,Can I send a money order from USPS as a business?,Yes. The USPS money order lets you put any nam...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,3,The AI’s answer is relevant to the user’s ques...
2,1 EIN doing business under multiple business n...,Yes. A single employer‑identification number i...,You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...,-1,The AI’s answer directly addresses the user’s ...
3,Applying for and receiving business credit,"To obtain business credit, follow these steps:...","""I'm afraid the great myth of limited liabilit...",Set up a meeting with the bank that handles yo...,Set up a meeting with the bank that handles yo...,4,The AI’s answer is highly relevant to the user...
4,401k Transfer After Business Closure,"When your employer shuts down, the 401(k) plan...",You should probably consult an attorney. Howev...,The time horizon for your 401K/IRA is essentia...,The time horizon for your 401K/IRA is essentia...,-1,The AI's answer directly addresses how to roll...


In [14]:
for idx in range(len(eval_input_lst)):
    retrieval_quality = pd.read_excel(all_config['eval_rq_output'][idx])
    answer_quality = pd.read_excel(all_config['eval_aq_output'][idx])
    all_df = retrieval_quality.merge(answer_quality[['query', 'answer_quality_score', 'aq_reasoning']], on='query')
    all_df.to_excel(all_config['eval_all_output'][idx], index=False)
    print(all_df.shape)
    display(all_df.head())

(30, 10)


,query,ai_answer,referenced_answer,retrieved_content,context,retrieval_quality_score,rq_reasoning,same_context,answer_quality_score,aq_reasoning
0,How to deposit a cheque issued to an associate...,To deposit a check that’s made out to an assoc...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,3,The retrieved content is identical to the cont...,True,4,The AI’s answer is fully relevant to the user’...
1,Can I send a money order from USPS as a business?,Yes. The USPS money order lets you put any nam...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,3,The retrieved content directly answers the use...,True,3,The AI’s answer is relevant to the user’s ques...
2,1 EIN doing business under multiple business n...,Yes. A single employer‑identification number i...,You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...,3,The retrieved content is identical to the prov...,True,-1,The AI’s answer directly addresses the user’s ...
3,Applying for and receiving business credit,"To obtain business credit, follow these steps:...","""I'm afraid the great myth of limited liabilit...",Set up a meeting with the bank that handles yo...,Set up a meeting with the bank that handles yo...,3,The retrieved content is identical to the cont...,True,4,The AI’s answer is highly relevant to the user...
4,401k Transfer After Business Closure,"When your employer shuts down, the 401(k) plan...",You should probably consult an attorney. Howev...,The time horizon for your 401K/IRA is essentia...,The time horizon for your 401K/IRA is essentia...,3,The retrieved content is a near‑duplicate of t...,False,-1,The AI's answer directly addresses how to roll...
